In [ ]:
import pandas as pd
import numpy as np
import wrds
import os, glob, re
from pathlib import Path
import yfinance as yf

In [ ]:
# Download side-information data (macro indicators) from Yahoo Finance
tickers = ["CL=F", "^GSPC", "^DJI", "^TNX", "^VIX"]
data = yf.download(tickers,
                   start="2015-01-01",
                   end="2024-01-01",
                   auto_adjust=True)["Close"]

data.to_csv("macro_raw.csv")

In [ ]:
# Making clena_returns, clean_sideinfo csv
SIDEINFO_CSV = "../data/macro_raw.csv"
CACHE_DIR = "../data/sp500_cache"

def _to_datetime_series(s):
    return pd.to_datetime(s, errors="coerce")

def find_date_col(df):
    cols = [c.lower() for c in df.columns]
    for key in ["date", "datetime", "time", "timestamp"]:
        if key in cols:
            return df.columns[cols.index(key)]
    c0 = df.columns[0]
    dt = _to_datetime_series(df[c0])
    if dt.notna().mean() > 0.8:
        return c0
    raise ValueError("No date column found.")

def standardize_date_index(df):
    date_col = find_date_col(df)
    df = df.copy()
    df[date_col] = _to_datetime_series(df[date_col]).dt.normalize()
    df = df.dropna(subset=[date_col]).sort_values(date_col)
    df = df.drop_duplicates(subset=[date_col], keep="last")
    df = df.set_index(date_col)
    return df

def is_ticker_filename(stem):
    return re.fullmatch(r"[A-Za-z0-9]{1,10}([.-][A-Za-z0-9]{1,5})?", stem) is not None

def load_macro_sideinfo(sideinfo_csv):

    raw = pd.read_csv(sideinfo_csv, low_memory=False)
    df = standardize_date_index(raw)

    numeric_cols = []
    for c in df.columns:
        x = pd.to_numeric(df[c], errors="coerce")
        if x.notna().mean() > 0.2:
            numeric_cols.append(c)

    s_df = df[numeric_cols].copy()
    s_df = s_df.apply(pd.to_numeric, errors="coerce")
    s_df = s_df.dropna(how="all")

    scale_factors = s_df.abs().median()
    scale_factors[scale_factors == 0] = 1.0
    s_df = s_df / scale_factors

    print("side info shape:", s_df.shape)
    return s_df

def load_returns(cache_dir):

    price_dict = {}

    for fp in sorted(glob.glob(os.path.join(cache_dir, "*.csv"))):
        stem = os.path.splitext(os.path.basename(fp))[0]
        if not is_ticker_filename(stem):
            continue

        ticker = stem.upper()

        try:
            df = pd.read_csv(fp, low_memory=False)
            df = standardize_date_index(df)

            price_col = None
            for c in ["Adj Close","Close","close","Price"]:
                if c in df.columns:
                    price_col = c
                    break

            if price_col is None:
                continue

            s = pd.to_numeric(df[price_col], errors="coerce")
            s = s.dropna()

            ret = s.pct_change()
            price_dict[ticker] = ret

        except:
            continue

    ret_df = pd.concat(price_dict.values(), axis=1)
    ret_df.columns = price_dict.keys()
    ret_df = ret_df.sort_index()

    print("returns raw shape:", ret_df.shape)
    return ret_df

def build_clean_panel(sideinfo_csv, cache_dir):

    s_df = load_macro_sideinfo(sideinfo_csv)
    ret_df = load_returns(cache_dir)

    common_dates = s_df.index.intersection(ret_df.index)

    s_df = s_df.loc[common_dates]
    ret_df = ret_df.loc[common_dates]

    print("✅ aligned shape:", ret_df.shape)

    return s_df, ret_df

def get_valid_assets_over_window(ret_df, end_date, window=252*2):

    if end_date not in ret_df.index:
        return []

    end_loc = ret_df.index.get_loc(end_date)

    if end_loc < window:
        return []

    window_slice = ret_df.iloc[end_loc-window:end_loc]

    valid_assets = window_slice.columns[window_slice.isna().sum() == 0]

    return list(valid_assets)

s_df, ret_df = build_clean_panel(SIDEINFO_CSV, CACHE_DIR)

s_df.to_csv("clean_sideinfo.csv")
ret_df.to_csv("clean_returns.csv")

print("Saved clean_sideinfo.csv")
print("Saved clean_returns.csv")

In [ ]:
"""USE IN LOCAL COMPUTER WITH WRDS ACCESS (or adapt to your data source)"""
# Using clean_returns.csv, match the membership based on the ticker and date by wrds

CLEAN_RETURNS_CSV = "../data/clean_returns.csv"   
OUT_EVENTS_CSV    = "../data/sp500_membership_events_wrds.csv"
OUT_MASK_CSV      = "../data/sp500_membership_mask_on_returns_dates.csv"

DATE_START = "2020-01-01"
DATE_END   = "2023-12-31"

rets = pd.read_csv(CLEAN_RETURNS_CSV, low_memory=False)

date_col = None
for c in ["Date", "date", "time", "Time", "datetime", "timestamp"]:
    if c in rets.columns:
        date_col = c
        break
if date_col is None:
    raise ValueError("No date column found in clean_returns.csv (expected Date/time/etc.)")

rets[date_col] = pd.to_datetime(rets[date_col], errors="coerce")
rets = rets.dropna(subset=[date_col]).sort_values(date_col)
rets = rets.rename(columns={date_col: "Date"})
rets = rets.set_index("Date")

tickers_in_data = [c for c in rets.columns]
dates_in_data = rets.index.unique()


DATE_START = pd.to_datetime(DATE_START)
DATE_END   = pd.to_datetime(DATE_END)

mask_dates = dates_in_data[(dates_in_data >= DATE_START) & (dates_in_data <= DATE_END)]
mask_dates = pd.DatetimeIndex(mask_dates).sort_values()

print("Returns tickers:", len(tickers_in_data))
print("Mask dates:", len(mask_dates), "|", mask_dates.min().date(), "to", mask_dates.max().date())

conn = wrds.Connection() 

sp500 = conn.raw_sql(
    f"""
    select permno, start, ending
    from crsp.msp500list
    where not (ending < '{DATE_START.date()}')
      and not (start  > '{DATE_END.date()}')
    """,
    date_cols=["start", "ending"]
)

print("Raw membership spells:", sp500.shape)

mse = conn.raw_sql(
    """
    select permno, ticker, namedt, nameendt
    from crsp.msenames
    """,
    date_cols=["namedt", "nameendt"]
)
mse["nameendt"] = mse["nameendt"].fillna(pd.to_datetime("today"))

sp500_full = sp500.merge(mse, on="permno", how="left")

overlap = (sp500_full["start"] <= sp500_full["nameendt"]) & (sp500_full["ending"] >= sp500_full["namedt"])
sp500_full = sp500_full.loc[overlap].copy()

sp500_full["ticker"] = sp500_full["ticker"].astype(str).str.strip().str.upper()
tickers_set = set([t.upper() for t in tickers_in_data])
sp500_full = sp500_full.loc[sp500_full["ticker"].isin(tickers_set)].copy()

sp500_full = sp500_full[["permno", "ticker", "start", "ending", "namedt", "nameendt"]].dropna(subset=["ticker"])
sp500_full.to_csv(OUT_EVENTS_CSV, index=False)
print("Saved membership spells:", OUT_EVENTS_CSV, "| rows:", len(sp500_full))

mask = pd.DataFrame(0, index=mask_dates, columns=[t.upper() for t in tickers_in_data], dtype=np.uint8)

for tkr, g in sp500_full.groupby("ticker"):
    tkr = tkr.upper()
    if tkr not in mask.columns:
        continue
    for _, row in g.iterrows():
        s = row["start"]
        e = row["ending"]

        s = max(s, mask_dates.min())
        e = min(e, mask_dates.max())
        if s > e:
            continue
        mask.loc[(mask.index >= s) & (mask.index <= e), tkr] = 1

mask.to_csv(OUT_MASK_CSV)
print("Saved membership mask:", OUT_MASK_CSV, "| shape:", mask.shape)

d = pd.to_datetime("2023-01-03")
if d in mask.index:
    print("S&P500 count on 2023-01-03:", int(mask.loc[d].sum()))
else:
    print("2023-01-03 not in mask dates (holiday or not in your returns index).")

def assets_2y_complete(rets_df, end_date, window=252*2):
    if end_date not in rets_df.index:
        return []
    loc = rets_df.index.get_loc(end_date)
    if loc < window:
        return []
    sl = rets_df.iloc[loc-window:loc]
    return sl.columns[sl.isna().sum()==0].tolist()

twoY = assets_2y_complete(rets, d)
if d in mask.index:
    sp_on_d = mask.columns[mask.loc[d].astype(bool)].tolist()
    inter = sorted(list(set([x.upper() for x in twoY]) & set(sp_on_d)))
    print("2y-complete count:", len(twoY))
    print("S&P500∩2y-complete count:", len(inter))

In [ ]:
# Monthly analysis of valid assets (S&P500 membership + 2y complete + full month complete)
RETURNS_CSV = "../data/clean_returns.csv"
MEMBERSHIP_MASK_CSV = "../data/sp500_membership_mask_on_returns_dates.csv" 

# ---- load returns ----
rets = pd.read_csv(RETURNS_CSV, low_memory=False)
rets["Date"] = pd.to_datetime(rets["Date"], errors="coerce")
rets = rets.dropna(subset=["Date"]).set_index("Date").sort_index()
rets.columns = rets.columns.astype(str).str.strip().str.upper()

# ---- load membership mask ----
m = pd.read_csv(MEMBERSHIP_MASK_CSV, low_memory=False)
date_col = "Date" if "Date" in m.columns else ("date" if "date" in m.columns else None)
if date_col is None:
    raise ValueError("membership mask CSV must have a Date/date column")

m[date_col] = pd.to_datetime(m[date_col], errors="coerce")
m = m.dropna(subset=[date_col]).rename(columns={date_col: "Date"}).set_index("Date").sort_index()
m.columns = m.columns.astype(str).str.strip().str.upper()

# ---- align (dates & columns) ----
common_dates = rets.index.intersection(m.index)
rets = rets.loc[common_dates]
m = m.loc[common_dates]

common_cols = rets.columns.intersection(m.columns)
rets = rets[common_cols]
m = m[common_cols].astype(int)

def first_trading_day_of_month(index, month_str):
    m0 = pd.to_datetime(month_str + "-01")
    m1 = m0 + pd.offsets.MonthBegin(1)
    days = index[(index >= m0) & (index < m1)]
    return None if len(days) == 0 else days[0]

def month_slice(index, month_str):
    m0 = pd.to_datetime(month_str + "-01")
    m1 = m0 + pd.offsets.MonthBegin(1)
    return index[(index >= m0) & (index < m1)]

def count_non_nan_on_date(df, d):
    row = df.loc[d]
    return int(row.notna().sum()), row

def assets_2y_complete_up_to(df, end_date, window=252*2):
    if end_date not in df.index:
        return []
    end_loc = df.index.get_loc(end_date)
    if end_loc < window:
        return []
    train_slice = df.iloc[end_loc-window:end_loc]
    return train_slice.columns[train_slice.isna().sum() == 0].tolist()

def sp500_assets_on_date(mask_df, d):
    if d not in mask_df.index:
        return []
    row = mask_df.loc[d]
    return row.index[row == 1].tolist()

def eligible_for_month_with_sp500(rets, mask, month_str, window=252*2):
    ms = first_trading_day_of_month(rets.index, month_str)
    if ms is None:
        return ms, [], [], []

    sp = sp500_assets_on_date(mask, ms)
    if len(sp) == 0:
        return ms, [], [], []

    valid_2y_all = assets_2y_complete_up_to(rets, ms, window=window)
    valid_2y_sp = sorted(list(set(valid_2y_all) & set(sp)))

    oos_days = month_slice(rets.index, month_str)
    oos_df = rets.loc[oos_days, valid_2y_sp] if len(valid_2y_sp) else rets.iloc[0:0]
    valid_month = oos_df.columns[oos_df.isna().sum() == 0].tolist()
    dropped_in_month = sorted(list(set(valid_2y_sp) - set(valid_month)))

    return ms, sp, valid_2y_sp, valid_month, dropped_in_month

month_str = "2023-01"
ms = first_trading_day_of_month(rets.index, month_str)

print("Month:", month_str)
print("First trading day:", ms.date() if ms is not None else None)

if ms is None:
    raise ValueError("No trading day found in this month within your data.")

n_non_nan, _ = count_non_nan_on_date(rets, ms)
print(f"\n[1] Returns non-NaN on {ms.date()}: {n_non_nan} / {rets.shape[1]}")

sp = sp500_assets_on_date(m, ms)
print(f"[1b] S&P500 members on {ms.date()} (within your ticker universe): {len(sp)}")

valid_2y_all = assets_2y_complete_up_to(rets, ms, window=252*2)
valid_2y_sp  = sorted(list(set(valid_2y_all) & set(sp)))
print(f"\n[2] 2y-complete tickers up to {ms.date()} (ALL): {len(valid_2y_all)}")
print(f"[2b] 2y-complete tickers up to {ms.date()} (S&P500@ms): {len(valid_2y_sp)}")

ms, sp, valid_2y_sp, valid_month_sp, dropped_in_month_sp = eligible_for_month_with_sp500(
    rets, m, month_str, window=252*2
)
print(f"\n[3] Month-eligible tickers (S&P500@ms + 2y-complete + full-month non-missing): {len(valid_month_sp)}")
print("    Dropped within month (among S&P500@ms & 2y-complete):", len(dropped_in_month_sp))
print("    Dropped tickers (first 30):", dropped_in_month_sp[:30])

missing_2y = sorted(list(set(sp) - set(valid_2y_sp)))
print(f"\n[Extra] S&P500@ms but NOT 2y-complete: {len(missing_2y)}")
print("        Examples (first 30):", missing_2y[:30])

In [ ]:
RETURNS_CSV = Path("../data/clean_returns.csv")
MASK_CSV    = Path("../data/sp500_membership_mask_on_returns_dates.csv")

def _ensure_time_col(df, prefer=("time","Time","date","Date","datetime","Datetime","timestamp","Timestamp")):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    if "time" not in df.columns:
        found = None
        for c in prefer:
            if c in df.columns:
                found = c
                break
        if found is None:
            found = df.columns[0]
        df = df.rename(columns={found: "time"})
    df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)
    return df

def _upper_cols(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    cols = ["time"] + [c for c in df.columns if c != "time"]
    df = df[cols]
    df.columns = [("time" if c=="time" else c.upper()) for c in df.columns]
    return df

rets = pd.read_csv(RETURNS_CSV, low_memory=False)
rets = _ensure_time_col(rets)
rets = _upper_cols(rets)

mask = pd.read_csv(MASK_CSV, low_memory=False)
mask = _ensure_time_col(mask)
mask = _upper_cols(mask)

common_dates = pd.Index(rets["time"]).intersection(mask["time"])
rets = rets[rets["time"].isin(common_dates)].sort_values("time").reset_index(drop=True)
mask = mask[mask["time"].isin(common_dates)].sort_values("time").reset_index(drop=True)

ret_cols  = [c for c in rets.columns if c != "time"]
mask_cols = [c for c in mask.columns if c != "time"]
tickers = sorted(list(set(ret_cols).intersection(mask_cols)))
if len(tickers) == 0:
    raise ValueError("No common tickers between returns and mask.")

rets = rets[["time"] + tickers].copy()
mask = mask[["time"] + tickers].copy()

rets[tickers] = rets[tickers].apply(pd.to_numeric, errors="coerce")
mask[tickers] = mask[tickers].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

R = rets.set_index("time")[tickers]
M = mask.set_index("time")[tickers]

print("Aligned shapes:", R.shape, M.shape)
print("Date range:", R.index.min().date(), "to", R.index.max().date())

nan_total = int(R.isna().sum().sum())
nan_rows  = int(R.isna().any(axis=1).sum())
nan_cols  = int(R.isna().any(axis=0).sum())

print("\n[CHECK 1] NaN in clean_returns (overall)")
print("Total NaN cells:", nan_total)
print("Rows(dates) with any NaN:", nan_rows, "/", len(R))
print("Cols(tickers) with any NaN:", nan_cols, "/", R.shape[1])

if nan_total > 0:
    top_cols = R.isna().sum().sort_values(ascending=False).head(20)
    print("\nTop 20 tickers by NaN count:")
    print(top_cols[top_cols > 0])

    top_rows = R.isna().sum(axis=1).sort_values(ascending=False).head(10)
    print("\nTop 10 dates by NaN count:")
    print(top_rows[top_rows > 0])

nan_under_mask = R.isna() & (M == 1)
bad_cells = int(nan_under_mask.sum().sum())
bad_dates = int(nan_under_mask.any(axis=1).sum())
bad_tickers = int(nan_under_mask.any(axis=0).sum())

print("\n[CHECK 2] NaN in returns where mask==1")
print("Bad cells (mask==1 & return is NaN):", bad_cells)
print("Bad dates count:", bad_dates)
print("Bad tickers count:", bad_tickers)

if bad_cells > 0:
    worst_dates = nan_under_mask.sum(axis=1).sort_values(ascending=False).head(10)
    print("\nWorst 10 dates (count of bad cells):")
    print(worst_dates[worst_dates > 0])

    worst_tickers = nan_under_mask.sum(axis=0).sort_values(ascending=False).head(20)
    print("\nWorst 20 tickers (count of bad cells):")
    print(worst_tickers[worst_tickers > 0])

    pairs = np.argwhere(nan_under_mask.values)
    print("\nExample bad (date, ticker) pairs (first 30):")
    for idx in pairs[:30]:
        d = nan_under_mask.index[idx[0]]
        t = nan_under_mask.columns[idx[1]]
        print(d.date(), t)

if bad_cells > 0:
    bad_long = (
        nan_under_mask.stack()
        .reset_index()
        .rename(columns={"level_0": "time", "level_1": "ticker", 0: "is_bad"})
    )
    bad_long = bad_long[bad_long["is_bad"]].drop(columns=["is_bad"])
    print("\nBad pairs dataframe head:")
    print(bad_long.head(20))